In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 넘겨준 npy 파일 로드
X_raw = np.load('X32.npy')
y = np.load('y32.npy')

print(f"전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

# 2. Train / Test 분할 (8:2 비율, 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

전체 데이터 형태: X=(2827876, 69), y=(2827876,)


In [4]:
from sklearn.preprocessing import RobustScaler

# 네트워크 데이터의 극단적인 이상치(Outlier)에 강한 RobustScaler 사용
scaler = RobustScaler()

# X_train에만 fit(기준 설정)과 transform(변환)을 동시에 적용
X_train_scaled = scaler.fit_transform(X_train)

# X_test는 절대 fit하지 않고, Train의 기준에 맞춰 transform(변환)만 적용!
X_test_scaled = scaler.transform(X_test)

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 랜덤 포레스트 모델 정의
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. 모델 학습
print("\n[Random Forest] 학습 시작...")
rf_model.fit(X_train_scaled, y_train)

# 3. 테스트 데이터 예측
y_pred = rf_model.predict(X_test_scaled)

# 4. 평가지표 계산
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# 5. 혼동 행렬 추출 및 오탐률(FPR) 계산 (0: 정상, 1: 공격 가정)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) 

# 6. 결과 저장
rf_performance = {
    "Model": "Random Forest",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
    "False Positive Rate (FPR)": fpr
}

# 7. 결과 출력
print("Random Forest 학습 및 평가 완료!")
print(f"- Accuracy : {acc:.4f}")
print(f"- Precision: {prec:.4f}")
print(f"- Recall   : {rec:.4f}")
print(f"- F1-Score : {f1:.4f}")
print(f"- 오탐률(FPR): {fpr*100:.4f}%")


[Random Forest] 학습 시작...
Random Forest 학습 및 평가 완료!
- Accuracy : 0.9990
- Precision: 0.9990
- Recall   : 0.9990
- F1-Score : 0.9990
- 오탐률(FPR): 0.0658%


In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

final_model = rf_model

# 2. 5겹(5-Fold) 교차 검증 설정 (클래스 비율 유지)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Random Forest 교차 검증 시작")

# 3. 교차 검증 수행 (평가지표: F1-Score)
# 참고: X_train_scaled와 y_train 전체를 넣어 5번 쪼개가며 검증
cv_scores = cross_val_score(final_model, X_train_scaled, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)

print("\n[교차 검증 결과]")
print(f"각 Fold별 F1-Score: {np.round(cv_scores, 4)}")
print(f"평균 F1-Score: {cv_scores.mean():.4f} (± {cv_scores.std():.4f})")

Random Forest 교차 검증 시작

[교차 검증 결과]
각 Fold별 F1-Score: [0.9989 0.9988 0.9988 0.9989 0.9988]
평균 F1-Score: 0.9988 (± 0.0000)


In [7]:

best_model = rf_model

# 기본 예측(predict) 대신 예측 확률(predict_proba)을 가져옴
y_pred_prob = best_model.predict_proba(X_test_scaled)[:, 1] # 클래스 1(공격)일 확률

# 기본 임계값은 0.5 (50%만 넘어도 공격으로 간주)
# 오탐을 줄이기 위해 '매우 확실할 때만(예: 85% 이상)' 공격으로 간주하도록 깐깐하게 변경
CUSTOM_THRESHOLD = 0.85 

# 새로운 임계값 적용
y_pred_strict = (y_pred_prob >= CUSTOM_THRESHOLD).astype(int)

# 깐깐해진 예측 결과로 다시 혼동 행렬 확인
new_cm = confusion_matrix(y_test, y_pred_strict)
tn, fp, fn, tp = new_cm.ravel()
new_fpr = fp / (fp + tn)

print(f"\n[임계값 {CUSTOM_THRESHOLD} 적용 후]")
print(f"오탐(False Positive) 건수: {fp}건")
print(f"새로운 오탐률(FPR): {new_fpr*100:.4f}%")


[임계값 0.85 적용 후]
오탐(False Positive) 건수: 229건
새로운 오탐률(FPR): 0.0504%


In [8]:
import joblib

final_best_model = rf_model

# 2. 모델과 스케일러를 파일로 저장
joblib.dump(final_best_model, 'ids_rf_model_ver2.joblib')
joblib.dump(scaler, 'ids_robust_scaler_ver2.joblib')

print("최종 모델 및 스케일러 저장 완료!")

최종 모델 및 스케일러 저장 완료!


In [9]:
import joblib

# 1. 파일 불러오기 (파일 경로를 정확히 맞춰주세요)
feature_columns = joblib.load('feature_columns (2).pkl')

# 2. 내용 확인하기
print(f"총 피처 개수: {len(feature_columns)}개")
print("\n[피처 이름 리스트 전체 보기]")
for i, col in enumerate(feature_columns):
    print(f"{i+1}. {col}")

총 피처 개수: 69개

[피처 이름 리스트 전체 보기]
1. Destination Port
2. Flow Duration
3. Total Fwd Packets
4. Total Backward Packets
5. Total Length of Fwd Packets
6. Total Length of Bwd Packets
7. Fwd Packet Length Max
8. Fwd Packet Length Min
9. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd URG Flags
33. Fwd Header Length
34. Bwd Header Length
35. Fwd Packets/s
36. Bwd Packets/s
37. Min Packet Length
38. Max Packet Length
39. Packet Length Mean
40. Packet Length Std
41. Packet Length Variance
42. FIN Flag Count
43. SYN Flag Count
44. RST Flag Count
45. PSH Flag Count
46. ACK Flag Count
47. URG Flag Coun

In [11]:
import joblib
import pandas as pd

# 1. 저장해둔 모델과 피처 리스트 파일 로드
loaded_model = joblib.load('ids_rf_model_ver2.joblib')
feature_columns = joblib.load('feature_columns (2).pkl')

# 2. 모델에서 피처 중요도(가중치) 추출
importances = loaded_model.feature_importances_

# 3. 중요도를 백분율(%)로 변환하여 데이터프레임으로 묶기
df_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance (%)': importances * 100
})

# 4. 내림차순 정렬 및 Top 10 추출
top10_features = df_importance.sort_values(by='Importance (%)', ascending=False).head(10)

print("[탐지 기여도 Top 10 피처]")
print(top10_features.to_markdown(index=False))

[탐지 기여도 Top 10 피처]
| Feature                |   Importance (%) |
|:-----------------------|-----------------:|
| Bwd Packet Length Std  |          6.89569 |
| Packet Length Variance |          6.45966 |
| Packet Length Std      |          6.30734 |
| Destination Port       |          6.21698 |
| Avg Bwd Segment Size   |          5.81258 |
| Bwd Packet Length Max  |          4.17416 |
| Packet Length Mean     |          3.97401 |
| Average Packet Size    |          3.96937 |
| Bwd Packet Length Mean |          3.69469 |
| Init_Win_bytes_forward |          3.32496 |
